In [1]:
import pandas as pd

import matplotlib.pyplot as plot
import matplotlib.dates as mdates
import seaborn as sns
import time

In [2]:
import pandas as pd

import matplotlib.pyplot as plot
import matplotlib.dates as mdates
import seaborn as sns
import time
from datetime import datetime, timedelta, date
import teradata
import businessdays as bzd
from pandasql import sqldf
import pygwalker as pyg
from pivottablejs import pivot_ui

import warnings
warnings.simplefilter('ignore')


start = time.time() 
system = "TERADBAWS.sicredi.net"
user = "P_USR_DL_GESTAO_RECURSOS"
passw = "JAW0b71329aadade35365afc6X"
udaExec = teradata.UdaExec (appName="TESTE", version="1.0", logConsole=False)
session = udaExec.connect(method="odbc", system=system,
        username=user, password=passw, Authentication="TDNEGO");



select_feriados = "Select DT from dl_Gestao_recursos.bas_cad_fer where tipo in ('3','2')"
feriados = pd.read_sql(select_feriados, session)
cal = bzd.Calendar(holidays=feriados.DT,weekdays=['Sunday','Saturday'])

data_inicial = cal.offset(datetime.now(),-60 )###DATA PADRÃO D-60
data_inicial = data_inicial.strftime('%Y-%m-%d') ###DATA PADRÃO D-90

data_final = cal.offset(datetime.now(),-1  )###DATA PADRÃO D-1
data_final = data_final.strftime('%Y-%m-%d') ###DATA PADRÃO D-1


In [3]:

Select = f"""select ajs.*, VCT.DT AS DT_VENCIMENTO, VCT.VL_PRECO_EXERC from dl_gestao_recursos.ajuste_bmf ajs
LEFT JOIN dl_Gestao_recursos.sac_fu_vcto vct on ajs.fuvct_fuatv_cd = vct.fuvct_fuatv_cd and ajs.fuvct_cd = vct.fuvct_cd
where AJS.dt between  '{data_inicial}' and '{data_final}'
and ajs.fuvct_fuatv_cd in ('OPD DOL','OPD WDO#') and sg_tipoopc = 'C'
 """


df_opd = pd.read_sql(Select, session) 
df_opd.DtPos = pd.to_datetime(df_opd.DT, format='%Y-%m-%d') 

pd.set_option('mode.chained_assignment', None)

In [4]:
display(df_opd)

,DT,FUVCT_FUATV_CD,FUVCT_CD,VL_PRECO,VL_AJUSTE_DIA_ANTERIOR,CONTRATO,DT_VENCIMENTO,VL_PRECO_EXERC
0,2024-08-14,OPD DOL,XSB8,3.626,4.295,DOLXSB8,2024-11-01,6525.0
1,2024-08-13,OPD DOL,XSBC,3.102,4.938,DOLXSBC,2024-11-01,6600.0
2,2024-08-22,OPD DOL,QT8S,99.657,76.093,DOLQT8S,2025-08-01,6700.0
3,2024-08-01,OPD DOL,FT9Y,14.091,9.737,DOLFT9Y,2025-01-02,7100.0
4,2024-08-22,OPD DOL,QT8N,127.131,100.139,DOLQT8N,2025-08-01,6500.0
...,...,...,...,...,...,...,...,...
56427,2024-06-13,OPD DOL,FY89,674.531,675.072,DOLFY89,2029-01-02,6400.0
56428,2024-06-18,OPD DOL,FY89,700.895,694.458,DOLFY89,2029-01-02,6400.0
56429,2024-08-15,OPD DOL,FY89,750.532,738.078,DOLFY89,2029-01-02,6400.0
56430,2024-08-23,OPD DOL,FY89,766.370,832.252,DOLFY89,2029-01-02,6400.0


In [ ]:
walker = pyg.walk(df_opd)

Box(children=(HTML(value='\n<div id="ifr-pyg-0006213a19dfa7a5n1VOT3fXJ5EFWc2p" style="height: auto">\n    <hea…

In [ ]:
############### P&L ##############

import matplotlib.pyplot as plt

# Dados da operação de put spread
expiration_date = '2025-01-02'
strike_price_long_put = 5700
strike_price_short_put = 6100
price_long_put = -254
price_short_put = 114
quantity = 1

# Preços do ativo subjacente (dólar) ao longo do tempo
underlying_prices = [5500, 5600, 5700, 5800, 5900,6000,6100,6200,6300]

# Função para calcular o P&L da operação de put spread
def calculate_pnl(underlying_price):
    pnl = (max(strike_price_long_put - underlying_price, 0) - max(strike_price_short_put - underlying_price, 0)) * quantity
    pnl -= (price_long_put - price_short_put) * quantity
    return pnl

# Listas para armazenar os valores de P&L e preços do ativo subjacente
pnl_values = []
underlying_price_values = []

# Loop para calcular o P&L para cada preço do ativo subjacente
for underlying_price in underlying_prices:
    pnl = calculate_pnl(underlying_price)
    pnl_values.append(pnl)
    underlying_price_values.append(underlying_price)

# Plotar o gráfico de P&L
plt.plot(underlying_price_values, pnl_values)
plt.xlabel('Preço do Ativo Subjacente')
plt.ylabel('P&L')
plt.title('Gráfico de P&L - Put Spread')
plt.grid(True)
plt.show()


In [ ]:
############ Volatildiade implicita ##############

""""
Para calcular a volatilidade implícita de um determinado vencimento de opções de dólar ao longo do tempo, você precisará dos seguintes inputs:

Dados históricos de preços do ativo subjacente (dólar) ao longo do tempo.
Preços de mercado das opções de dólar para diferentes vencimentos.
Taxa de juros livre de risco.
Método de cálculo da volatilidade implícita (por exemplo, modelo Black-Scholes).
Com essas informações, você pode usar a biblioteca py_vollib em Python para calcular a volatilidade implícita. Aqui está um exemplo de código que pode ajudar:

Neste exemplo, você precisará fornecer os dados históricos de preços do ativo subjacente (dólar) ao longo do tempo em um arquivo CSV chamado 'historicalprices.csv' e os preços de mercado das opções de dólar para diferentes vencimentos em um arquivo CSV chamado 'optionprices.csv'.

Você também precisará ajustar o código para obter os dados relevantes das opções, como preço, strike, tipo de opção, etc., e calcular o tempo até o vencimento (timetoexpiration) com base na data atual e na data de vencimento.

Lembre-se de que a volatilidade implícita é um conceito complexo e requer uma compreensão sólida dos modelos de precificação de opções. É sempre recomendável consultar um especialista financeiro ou um profissional do mercado de capitais para obter orientações adequadas ao lidar com opções e volatilidade implícita.

""""

import py_vollib.black_scholes.implied_volatility as iv
import pandas as pd

# Dados históricos de preços do ativo subjacente (dólar) ao longo do tempo
historical_prices = pd.read_csv('historical_prices.csv')

# Preços de mercado das opções de dólar para diferentes vencimentos
option_prices = pd.read_csv('option_prices.csv')

# Taxa de juros livre de risco
risk_free_rate = 0.05

# Método de cálculo da volatilidade implícita (por exemplo, modelo Black-Scholes)

# Loop para calcular a volatilidade implícita para cada vencimento
for expiration_date in option_prices['expiration_date'].unique():
    # Filtrar os dados de opções para o vencimento específico
    options = option_prices[option_prices['expiration_date'] == expiration_date]

    # Loop para calcular a volatilidade implícita para cada opção
    for index, option in options.iterrows():
        # Obter os dados relevantes da opção (preço, strike, tipo de opção, etc.)
        option_price = option['option_price']
        strike_price = option['strike_price']
        option_type = option['option_type']

        # Calcular a volatilidade implícita usando o modelo Black-Scholes
        implied_volatility = iv.implied_volatility(option_price, underlying_price, strike_price, time_to_expiration, risk_free_rate, option_type)

        # Armazenar ou imprimir a volatilidade implícita calculada

In [ ]:
############# Probabildiade Implicita ###############

""""
Para calcular a probabilidade implícita em um determinado strike em um determinado vencimento ao longo de um período de tempo, você precisará dos seguintes inputs:

Dados históricos de preços do ativo subjacente (dólar) ao longo do tempo.
Preços de mercado das opções de dólar para diferentes vencimentos.
Taxa de juros livre de risco.
Método de cálculo da probabilidade implícita (por exemplo, modelo Black-Scholes).
Aqui está um exemplo de código que pode ajudar:

Neste exemplo, você precisará fornecer os dados históricos de preços do ativo subjacente (dólar) ao longo do tempo em um arquivo CSV chamado 'historicalprices.csv' e os preços de mercado das opções de dólar para diferentes vencimentos em um arquivo CSV chamado 'optionprices.csv'.

Você também precisará ajustar o código para obter os dados relevantes das opções, como preço, strike, tipo de opção, etc., e calcular o tempo até o vencimento (timetoexpiration) com base na data atual e na data de vencimento.

Lembre-se de que a probabilidade implícita é uma estimativa subjetiva e depende do modelo de precificação de opções utilizado. É sempre recomendável consultar um especialista financeiro ou um profissional do mercado de capitais para obter orientações adequadas ao lidar com opções e probabilidade implícita.
""""

import py_vollib.black_scholes.implied_volatility as iv
import py_vollib.black_scholes as bs
import pandas as pd

# Dados históricos de preços do ativo subjacente (dólar) ao longo do tempo
historical_prices = pd.read_csv('historical_prices.csv')

# Preços de mercado das opções de dólar para diferentes vencimentos
option_prices = pd.read_csv('option_prices.csv')

# Taxa de juros livre de risco
risk_free_rate = 0.05

# Método de cálculo da probabilidade implícita (por exemplo, modelo Black-Scholes)

# Loop para calcular a probabilidade implícita para cada vencimento
for expiration_date in option_prices['expiration_date'].unique():
    # Filtrar os dados de opções para o vencimento específico
    options = option_prices[option_prices['expiration_date'] == expiration_date]

    # Loop para calcular a probabilidade implícita para cada opção
    for index, option in options.iterrows():
        # Obter os dados relevantes da opção (preço, strike, tipo de opção, etc.)
        option_price = option['option_price']
        strike_price = option['strike_price']
        option_type = option['option_type']

        # Calcular a volatilidade implícita usando o modelo Black-Scholes
        implied_volatility = iv.implied_volatility(option_price, underlying_price, strike_price, time_to_expiration, risk_free_rate, option_type)

        # Calcular a probabilidade implícita usando o modelo Black-Scholes
        implied_probability = bs.black_scholes(option_type, underlying_price, strike_price, time_to_expiration, risk_free_rate, implied_volatility)

        # Armazenar ou imprimir a probabilidade implícita calculada